In [1]:
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from scipy import ndimage
import matplotlib.pyplot as plt
import os

%matplotlib inline

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Device: {device}")

BASE_PATH = '/Users/zabir/Downloads/semi-AD/semi-AD'
MEMORY_BANK_PATH = os.path.expanduser('~/Documents/SemiAD/memory_banks')
REPORTS_PATH = os.path.expanduser('~/Documents/SemiAD/reports')

SUBSETS = {
    'ball_side': f'{BASE_PATH}/IC_substrate/ball_side',
    'chip_side': f'{BASE_PATH}/IC_substrate/chip_side',
    'wafer':     f'{BASE_PATH}/Wafer/patterned'
}

print("Paths set.")

Device: mps
Paths set.


In [2]:
# Multi-scale feature extractor — same architecture as notebook 02
class MultiScaleFeatureExtractor(torch.nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.early = torch.nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu,
            backbone.maxpool, backbone.layer1, backbone.layer2
        )
        self.late = backbone.layer3

    def forward(self, x):
        layer2_out = self.early(x)
        layer3_out = self.late(layer2_out)
        layer2_resized = F.interpolate(layer2_out, size=layer3_out.shape[2:], mode='bilinear', align_corners=False)
        return torch.cat([layer2_resized, layer3_out], dim=1)

backbone = models.resnet18(pretrained=True)
feature_extractor = MultiScaleFeatureExtractor(backbone).to(device)
for param in feature_extractor.parameters():
    param.requires_grad = False
feature_extractor.eval()
print("Feature extractor ready.")

# Load images from a folder into a tensor
def load_images(folder, transform):
    images = []
    filenames = sorted([f for f in os.listdir(folder)
                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))])
    for fname in tqdm(filenames, desc=f'Loading {os.path.basename(folder)}'):
        img = Image.open(f'{folder}/{fname}').convert('RGB')
        images.append(transform(img))
    return torch.stack(images), filenames

# Load ground truth masks and binarize
def load_masks(mask_folder, img_size=256):
    masks = []
    mask_files = sorted([f for f in os.listdir(mask_folder)
                        if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))])
    for fname in mask_files:
        mask = np.array(Image.open(f'{mask_folder}/{fname}').convert('L').resize(
            (img_size, img_size), Image.NEAREST))
        masks.append((mask > 0).astype(np.uint8))
    return np.array(masks)

# Compute per-patch anomaly maps by comparing to memory bank
def compute_anomaly_maps(imgs, model, memory_bank, batch_size=32, img_size=256):
    all_maps = []
    memory_bank_gpu = memory_bank.to(device)
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(imgs), batch_size), desc='Computing anomaly maps'):
            batch = imgs[i:i+batch_size].to(device)
            features = model(batch)
            b, c, h, w = features.shape
            features = features.permute(0, 2, 3, 1).reshape(b, h*w, c)
            for img_features in features:
                distances = torch.cdist(img_features, memory_bank_gpu)
                min_distances, _ = distances.min(dim=1)
                side = int(min_distances.shape[0] ** 0.5)
                anomaly_map = min_distances.reshape(side, side).cpu().numpy()
                anomaly_map = np.array(Image.fromarray(anomaly_map).resize(
                    (img_size, img_size), Image.BILINEAR))
                all_maps.append(anomaly_map)
    return np.array(all_maps)

/Users/zabir/Documents/SemiAD/semiad_env/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/zabir/Documents/SemiAD/semiad_env/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Feature extractor ready.


In [ ]:
# Load all three memory banks from disk
memory_banks = {}

for name in SUBSETS.keys():
    path = f'{MEMORY_BANK_PATH}/{name}_memory_bank.pt'
    memory_banks[name] = torch.load(path, map_location='cpu')
    print(f"Loaded {name}: {memory_banks[name].shape}")


Loaded ball_side: torch.Size([142899, 384])
Loaded chip_side: torch.Size([175872, 384])
Loaded wafer: torch.Size([113230, 384])

All memory banks loaded.


In [4]:
# Load test images and compute raw anomaly maps for all subsets
raw_maps = {}
gt_masks = {}
test_imgs_all = {}

for name, base in SUBSETS.items():
    img_size = 200 if name == 'wafer' else 256

    subset_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    print(f"\n--- {name} ---")
    test_defect_s, _ = load_images(f'{base}/single_set/test/defect', subset_transform)
    gt_masks[name] = load_masks(f'{base}/single_set/ground_truth/defect', img_size=img_size)
    raw_maps[name] = compute_anomaly_maps(test_defect_s, feature_extractor, memory_banks[name], img_size=img_size)
    test_imgs_all[name] = test_defect_s

    print(f"Anomaly maps shape: {raw_maps[name].shape}")
    print(f"GT masks shape:     {gt_masks[name].shape}")


--- ball_side ---


Computing anomaly maps: 100%|██████████| 11/11 [00:15<00:00,  1.42s/it]


Anomaly maps shape: (324, 256, 256)
GT masks shape:     (324, 256, 256)

--- chip_side ---


Computing anomaly maps: 100%|██████████| 5/5 [00:07<00:00,  1.57s/it]


Anomaly maps shape: (147, 256, 256)
GT masks shape:     (147, 256, 256)

--- wafer ---


Computing anomaly maps: 100%|██████████| 11/11 [00:09<00:00,  1.16it/s]

Anomaly maps shape: (336, 200, 200)
GT masks shape:     (336, 200, 200)


In [5]:
# Apply Gaussian smoothing to raw anomaly maps
# sigma controls blur strength — higher = smoother but risks washing out tiny defects
SIGMA = 4

smoothed_maps = {}

for name, maps in raw_maps.items():
    smoothed = np.array([
        ndimage.gaussian_filter(m, sigma=SIGMA) for m in maps
    ])
    smoothed_maps[name] = smoothed
    print(f"{name}: smoothed shape {smoothed.shape}, "
          f"value range [{smoothed.min():.4f}, {smoothed.max():.4f}]")

print(f"\nGaussian smoothing applied (sigma={SIGMA}).")

ball_side: smoothed shape (324, 256, 256), value range [0.4599, 6.5600]
chip_side: smoothed shape (147, 256, 256), value range [0.4678, 6.7284]
wafer: smoothed shape (336, 200, 200), value range [0.0000, 6.2168]

Gaussian smoothing applied (sigma=4).


In [6]:
# Baseline results from notebook 02 — hardcoded for comparison
baseline = {
    'ball_side': {'image_auroc': 0.8273, 'pixel_auroc': 0.9887},
    'chip_side': {'image_auroc': 0.7815, 'pixel_auroc': 0.9485},
    'wafer':     {'image_auroc': 0.9111, 'pixel_auroc': 0.9829},
}

# Evaluate smoothed maps
smoothed_results = {}

for name, maps in smoothed_maps.items():
    # Image-level score = max pixel score per image
    image_scores = maps.reshape(len(maps), -1).max(axis=1)

    # Load good test images for image-level AUROC
    img_size = 200 if name == 'wafer' else 256
    subset_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    test_good_s, _ = load_images(f'{SUBSETS[name]}/single_set/test/good', subset_transform)
    good_maps_s = compute_anomaly_maps(test_good_s, feature_extractor, memory_banks[name], img_size=img_size)
    good_maps_smoothed = np.array([ndimage.gaussian_filter(m, sigma=SIGMA) for m in good_maps_s])
    good_scores = good_maps_smoothed.reshape(len(good_maps_smoothed), -1).max(axis=1)

    # Image AUROC
    labels = np.concatenate([np.zeros(len(good_scores)), np.ones(len(image_scores))])
    scores = np.concatenate([good_scores, image_scores])
    img_auroc = roc_auc_score(labels, scores)

    # Pixel AUROC
    pixel_auroc = roc_auc_score(gt_masks[name].flatten(), maps.flatten())

    smoothed_results[name] = {
        'image_auroc': img_auroc,
        'pixel_auroc': pixel_auroc,
    }

# Print comparison table
print(f"Gaussian sigma = {SIGMA}\n")
print(f"{'Subset':<12} {'Metric':<14} {'Baseline':>10} {'Smoothed':>10} {'Delta':>8}")
print("-" * 58)
for name in SUBSETS.keys():
    b = baseline[name]
    s = smoothed_results[name]
    for metric in ['image_auroc', 'pixel_auroc']:
        delta = s[metric] - b[metric]
        delta_str = f"+{delta:.4f}" if delta >= 0 else f"{delta:.4f}"
        print(f"{name:<12} {metric:<14} {b[metric]:>10.4f} {s[metric]:>10.4f} {delta_str:>8}")
    print()

Computing anomaly maps: 100%|██████████| 150/150 [02:15<00:00,  1.10it/s]


Gaussian sigma = 4

Subset       Metric           Baseline   Smoothed    Delta
----------------------------------------------------------
ball_side    image_auroc        0.8273     0.7673  -0.0600
ball_side    pixel_auroc        0.9887     0.9884  -0.0003

chip_side    image_auroc        0.7815     0.7337  -0.0478
chip_side    pixel_auroc        0.9485     0.9462  -0.0023

wafer        image_auroc        0.9111     0.8904  -0.0207
wafer        pixel_auroc        0.9829     0.9829  -0.0000

